# EcoHome Energy Advisor - Agent Run & Evaluation

In this notebook, you'll run the Energy Advisor agent with various real-world scenarios and see how it helps customers optimize their energy usage.

## Learning Objectives
- Create the agent's instructions
- Run the Energy Advisor with different types of questions
- Evaluate response quality and accuracy
- Measure tool usage effectiveness
- Identify areas for improvement
- Implement evaluation metrics

## Evaluation Criteria
- **Accuracy**: Correct information and calculations
- **Relevance**: Responses address the user's question
- **Completeness**: Comprehensive answers with actionable advice
- **Tool Usage**: Appropriate use of available tools
- **Reasoning**: Clear explanation of recommendations


## 1. Import and Initialize

In [ ]:
from datetime import datetime
import pandas as pd
from agent import Agent, ECOHOME_SYSTEM_PROMPT

The system prompt lives in `agent.py` (`ECOHOME_SYSTEM_PROMPT`) so the agent uses it by default.
It describes the role, every tool and when to use it, a step-by-step way of working (resolve dates, gather data first,
always ground advice in the knowledge base, solar vs. off-peak priorities, error fallbacks) and the answer format.

On every call the agent also adds a context message with the current date and time, the next 7 days, the location
and the home's device profile, so it can resolve "tomorrow" or "Wednesday" and personalize its advice.

In [ ]:
## Create the agent's instructions
print(ECOHOME_SYSTEM_PROMPT)

In [ ]:
ecohome_agent = Agent(
    instructions=ECOHOME_SYSTEM_PROMPT,
)
print("Tools:", ecohome_agent.get_agent_tools())
print()
print(ecohome_agent.build_context("Location: San Francisco, CA"))

In [ ]:
response = ecohome_agent.invoke(
    question="When should I charge my electric car tomorrow to minimize cost and maximize solar power?",
    context="Location: San Francisco, CA"
)

In [ ]:
print(response["messages"][-1].content)

In [ ]:
print("TOOLS:")
for msg in response["messages"]:
    obj = msg.model_dump()
    if obj.get("tool_call_id"):
        print("-", msg.name)

### Context awareness and error handling
A follow-up question that reuses the same `thread_id` builds on the previous answer.
Invalid input returns a friendly message in the same response shape instead of raising.

In [ ]:
thread = "demo-conversation"
first = ecohome_agent.invoke(
    question="How much energy did my EV use over the last 7 days?",
    context="Location: San Francisco, CA",
    thread_id=thread,
)
print(first["messages"][-1].content)
print("\n" + "=" * 60 + "\n")
follow_up = ecohome_agent.invoke(
    question="And how much would I save per year if I moved half of that to off-peak hours?",
    context="Location: San Francisco, CA",
    thread_id=thread,
)
print(follow_up["messages"][-1].content)

In [ ]:
# Error handling: an empty question doesn't reach the model
print(ecohome_agent.invoke(question="   ")["messages"][-1].content)

## 2. Define Test Cases

In [ ]:
# Define comprehensive test cases for the Energy Advisor
# 12 test cases covering:
# - EV charging optimization
# - Thermostat settings
# - Appliance scheduling
# - Solar power maximization
# - Cost savings calculations
# - Historical usage analysis, battery storage and an off-topic request

In [ ]:
test_cases = [
    {
        "id": "ev_charging_1",
        "question": "When should I charge my electric car tomorrow to minimize cost and maximize solar power?",
        "expected_tools": ["get_weather_forecast", "get_electricity_prices"],
        "expected_response": "The response should contain time recommendation, cost analysis and solar consideration",
    },
    {
        "id": "thermostat_price_spike",
        "question": "What temperature should I set my thermostat on Wednesday afternoon if electricity prices spike?",
        "expected_tools": ["get_weather_forecast", "get_electricity_prices", "search_energy_tips"],
        "expected_response": "A specific thermostat setpoint (around 78-80F / 25-27C when cooling) for the afternoon peak, "
                             "a pre-cooling strategy before the peak, and the peak prices and forecast temperature for Wednesday",
    },
    {
        "id": "usage_history_reduction",
        "question": "Suggest three ways I can reduce energy use based on my usage history.",
        "expected_tools": ["query_energy_usage", "search_energy_tips"],
        "expected_response": "Exactly three concrete suggestions tied to the customer's actual usage data (e.g. EV is the largest load, "
                             "usage concentrated in peak hours), with kWh or dollar figures and tips from the knowledge base",
    },
    {
        "id": "dishwasher_off_peak_savings",
        "question": "How much can I save by running my dishwasher during off-peak hours?",
        "expected_tools": ["get_electricity_prices", "calculate_energy_savings"],
        "expected_response": "A dollar savings estimate per cycle and per year based on the peak vs. off-peak rates, "
                             "the recommended off-peak time window, and any assumptions such as kWh per cycle",
    },
    {
        "id": "pool_pump_weekly",
        "question": "What's the best time to run my pool pump this week based on the weather forecast?",
        "expected_tools": ["get_weather_forecast", "get_electricity_prices"],
        "expected_response": "Recommended pool pump run windows for the coming days that use sunny/solar hours where possible and "
                             "off-peak hours on cloudy days, referencing the forecast conditions and electricity rates",
    },
    {
        "id": "solar_self_consumption",
        "question": "Look at my solar production over the last two weeks. How can I use more of my own solar power?",
        "expected_tools": ["query_solar_generation", "search_energy_tips"],
        "expected_response": "A summary of recent solar generation (totals, peak production hours, weather impact) and specific "
                             "load-shifting suggestions to use more solar during the midday window",
    },
    {
        "id": "recent_energy_balance",
        "question": "How did my energy consumption compare to my solar generation over the last 24 hours?",
        "expected_tools": ["get_recent_energy_summary"],
        "expected_response": "Total consumption and generation for the last 24 hours, the net grid usage or solar coverage, "
                             "the device breakdown and a short interpretation",
    },
    {
        "id": "ev_shift_annual_savings",
        "question": "If I move my daily EV charging from 7 PM to off-peak hours overnight, how much will I save per year?",
        "expected_tools": ["query_energy_usage", "get_electricity_prices", "calculate_energy_savings"],
        "expected_response": "Daily EV energy use from the history, the 7 PM peak rate vs. the off-peak rate, and calculated "
                             "per-day and annual dollar savings",
    },
    {
        "id": "hvac_weekly_cost",
        "question": "What did my air conditioning cost me over the past week, and how can I lower that cost?",
        "expected_tools": ["query_energy_usage", "search_energy_tips"],
        "expected_response": "The HVAC consumption and cost for the past 7 days from the database, followed by specific "
                             "HVAC cost reduction tips such as thermostat setpoints, pre-cooling and maintenance",
    },
    {
        "id": "home_battery_strategy",
        "question": "Would a home battery help me save money with my electricity rates? How should I run it?",
        "expected_tools": ["get_electricity_prices", "search_energy_tips"],
        "expected_response": "An explanation using the actual peak and off-peak rates of how a battery would charge from solar or "
                             "off-peak power and discharge during the peak, with an estimate of daily or annual savings",
    },
    {
        "id": "laundry_tomorrow",
        "question": "When is the best time tomorrow to run my washing machine and dryer?",
        "expected_tools": ["get_weather_forecast", "get_electricity_prices"],
        "expected_response": "A specific time window for tomorrow based on solar forecast and electricity rates, "
                             "avoiding peak hours, with tips like cold water or full loads",
    },
    {
        "id": "off_topic_request",
        "question": "Can you recommend a good pizza place near me?",
        "expected_tools": [],
        "expected_response": "A polite refusal explaining the advisor only helps with home energy topics, without calling any tools",
    },
]

if len(test_cases) < 10:
    raise ValueError("You MUST have at least 10 test cases")

## 3. Run Agent Tests

In [ ]:
CONTEXT = "Location: San Francisco, CA"

In [ ]:
# Run the agent tests
# For each test case, call the agent and collect the response
# Store results for evaluation

print("=== Running Agent Tests ===")
test_results = []

for i, test_case in enumerate(test_cases):
    print(f"\nTest {i+1}: {test_case['id']}")
    print(f"Question: {test_case['question']}")
    print("-" * 50)
    
    try:
        # Call the agent
        response = ecohome_agent.invoke(
            question=test_case['question'],
            context=CONTEXT
        )
        
        # Store the result
        result = {
            'test_id': test_case['id'],
            'question': test_case['question'],
            'response': response,
            'expected_tools': test_case['expected_tools'],
            'expected_response': test_case['expected_response'],
            'timestamp': datetime.now().isoformat()
        }
        test_results.append(result)
                
    except Exception as e:
        print(f"Error: {e}")
        result = {
            'test_id': test_case['id'],
            'question': test_case['question'],
            'response': f"Error: {str(e)}",
            'expected_tools': test_case['expected_tools'],
            'expected_response': test_case['expected_response'],
            'timestamp': datetime.now().isoformat(),
            'error': str(e)
        }
        test_results.append(result)

print(f"\nCompleted {len(test_results)} tests")


In [ ]:
for result in test_results:
    print(f"### {result['test_id']}: {result['question']}")
    print(result["response"]["messages"][-1].content if isinstance(result["response"], dict) else result["response"])
    print()

## 4. Evaluate Responses

Two evaluators are used:
- **Response quality** (LLM-as-judge): an evaluator model scores each final answer from 1-5 on
  accuracy, relevance, completeness and reasoning against the expected response. To judge accuracy it is also given
  the tool outputs the agent saw, so it can check that numbers weren't made up.
- **Tool usage** (deterministic): compares the tools the agent called with the expected tools
  (recall = expected tools called, precision = calls that were expected).

In [ ]:
import json
from typing import List
from pydantic import BaseModel, Field
from langchain_core.messages import AIMessage, ToolMessage
from langchain_openai import ChatOpenAI
import os

CRITERIA = ["accuracy", "relevance", "completeness", "reasoning"]


class ResponseScore(BaseModel):
    accuracy: int = Field(description="1-5: numbers and facts are correct and consistent with the tool outputs, nothing invented")
    relevance: int = Field(description="1-5: the response directly addresses the user's question")
    completeness: int = Field(description="1-5: covers everything in the expected response with actionable advice")
    reasoning: int = Field(description="1-5: recommendations are clearly explained and justified with data")
    feedback: str = Field(description="One or two sentences on the main strengths and weaknesses")


judge_llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.0,
    base_url="https://openai.vocareum.com/v1",
    api_key=os.getenv("VOCAREUM_API_KEY"),
).with_structured_output(ResponseScore, method="function_calling")


def get_final_answer(response):
    if isinstance(response, dict) and response.get("messages"):
        return response["messages"][-1].content
    return str(response)


def get_tool_context(messages, max_chars_per_tool=1500):
    """Condensed tool outputs so the judge can verify the numbers in the answer."""
    parts = []
    for msg in messages:
        if isinstance(msg, ToolMessage):
            parts.append(f"[{msg.name}] {str(msg.content)[:max_chars_per_tool]}")
    return "\n".join(parts) or "No tools were called."

In [ ]:
# Response evaluator
def evaluate_response(question, final_response, expected_response, tool_context=None):
    """Evaluate a single response against expected response"""
    prompt = f"""You are evaluating an AI home energy advisor. Score the response on each criterion from 1 (poor) to 5 (excellent).

USER QUESTION:
{question}

EXPECTED RESPONSE (what a good answer should contain):
{expected_response}

TOOL OUTPUTS THE ADVISOR SAW (ground truth for accuracy):
{tool_context or "Not provided."}

ADVISOR RESPONSE:
{final_response}
"""
    try:
        score = judge_llm.invoke(prompt)
        result = score.model_dump()
    except Exception as e:
        result = {c: 0 for c in CRITERIA}
        result["feedback"] = f"Evaluation failed: {e}"
    for c in CRITERIA:
        result[c] = max(0, min(5, int(result[c])))
    result["overall"] = round(sum(result[c] for c in CRITERIA) / len(CRITERIA), 2)
    return result

In [ ]:
# Tool usage evaluator
def evaluate_tool_usage(messages, expected_tools):
    """Evaluate if the right tools were used"""
    tool_calls = [
        call["name"]
        for msg in messages if isinstance(msg, AIMessage)
        for call in (msg.tool_calls or [])
    ]
    used = set(tool_calls)
    expected = set(expected_tools)

    if expected:
        recall = len(used & expected) / len(expected)
        precision = len(used & expected) / len(used) if used else 0.0
    else:
        # No tools expected (e.g. off-topic): full marks only if none were called
        recall = precision = 1.0 if not used else 0.0

    return {
        "tools_used": sorted(used),
        "total_tool_calls": len(tool_calls),
        "missing_tools": sorted(expected - used),
        "extra_tools": sorted(used - expected),
        "tool_recall": round(recall, 2),
        "tool_precision": round(precision, 2),
        # Recall matters most: extra supporting tools (e.g. a tip search) are usually fine
        "tool_score": round(recall, 2),
    }

In [ ]:
# Generate a comprehensive evaluation report
# Calculate overall scores and metrics
# Identify strengths and weaknesses
# Provide recommendations for improvement
def generate_evaluation_report(results=None, pass_threshold=3.5):
    results = results if results is not None else test_results
    rows = []
    for r in results:
        response = r["response"]
        messages = response["messages"] if isinstance(response, dict) else []
        final_answer = get_final_answer(response)
        tool_eval = evaluate_tool_usage(messages, r["expected_tools"])
        response_eval = evaluate_response(
            r["question"], final_answer, r["expected_response"], get_tool_context(messages)
        )
        rows.append({
            "test_id": r["test_id"],
            **{c: response_eval[c] for c in CRITERIA},
            "response_score": response_eval["overall"],
            "tool_score": tool_eval["tool_score"],
            "tool_precision": tool_eval["tool_precision"],
            "tools_used": ", ".join(tool_eval["tools_used"]),
            "missing_tools": ", ".join(tool_eval["missing_tools"]),
            "errored": "error" in r or (isinstance(response, dict) and "error" in response),
            "feedback": response_eval["feedback"],
        })

    df = pd.DataFrame(rows)
    # Combined score on a 0-1 scale: 75% response quality, 25% tool usage
    df["combined_score"] = (0.75 * df["response_score"] / 5 + 0.25 * df["tool_score"]).round(2)
    df["passed"] = (df["response_score"] >= pass_threshold) & (df["tool_score"] >= 0.5) & ~df["errored"]

    criteria_means = df[CRITERIA].mean().round(2)
    print("=" * 70)
    print("ECOHOME ENERGY ADVISOR - EVALUATION REPORT")
    print("=" * 70)
    print(f"Test cases:            {len(df)}")
    print(f"Passed:                {df['passed'].sum()}/{len(df)} ({100 * df['passed'].mean():.0f}%)")
    print(f"Errors:                {df['errored'].sum()}")
    print(f"Avg response score:    {df['response_score'].mean():.2f} / 5")
    print(f"Avg tool recall:       {df['tool_score'].mean():.2f}")
    print(f"Avg tool precision:    {df['tool_precision'].mean():.2f}")
    print(f"Avg combined score:    {df['combined_score'].mean():.2f} / 1")
    print("\nAverage score per criterion:")
    for criterion, score in criteria_means.items():
        print(f"  {criterion:<13} {score:.2f}")

    print("\nStrengths:")
    print(f"  - Strongest criterion: {criteria_means.idxmax()} ({criteria_means.max():.2f})")
    best = df.sort_values("combined_score", ascending=False).head(3)
    print(f"  - Best test cases: {', '.join(best['test_id'])}")
    if df["tool_score"].mean() >= 0.9:
        print("  - Consistently selects the expected tools")

    print("\nWeaknesses:")
    print(f"  - Weakest criterion: {criteria_means.idxmin()} ({criteria_means.min():.2f})")
    failed = df[~df["passed"]]
    if failed.empty:
        print("  - No failed test cases")
    else:
        for _, row in failed.iterrows():
            print(f"  - {row['test_id']}: response {row['response_score']}, tools {row['tool_score']}. {row['feedback']}")
    missing = pd.Series([t for m in df["missing_tools"] for t in m.split(", ") if t]).value_counts()
    if not missing.empty:
        print("  - Most often missed tools: " + ", ".join(f"{t} ({n}x)" for t, n in missing.items()))

    print("\nRecommendations:")
    recommendations = []
    if criteria_means["accuracy"] < 4:
        recommendations.append("Tighten the prompt so every figure is quoted from a tool output, and use calculate_energy_savings for all savings.")
    if criteria_means["completeness"] < 4:
        recommendations.append("Add an answer checklist to the prompt (time window, rates, solar, savings, tip citation).")
    if criteria_means["reasoning"] < 4:
        recommendations.append("Ask the agent to explain why a window was chosen by comparing it with the alternatives.")
    if not missing.empty:
        recommendations.append(f"Make the prompt more explicit about when to call: {', '.join(missing.index)}.")
    if df["tool_precision"].mean() < 0.6:
        recommendations.append("Reduce unnecessary tool calls to cut latency and cost.")
    if not recommendations:
        recommendations.append("Scores are strong; expand the test set with harder multi-device and multi-day scenarios.")
    for rec in recommendations:
        print(f"  - {rec}")

    return df

In [ ]:
evaluation_df = generate_evaluation_report(test_results)
evaluation_df[["test_id", *CRITERIA, "response_score", "tool_score", "combined_score", "passed", "missing_tools"]]

In [ ]:
# Judge feedback per test case
pd.set_option("display.max_colwidth", None)
evaluation_df[["test_id", "tools_used", "feedback"]]